In [1]:
# 1. ติดตั้งไลบรารี
!pip install pytorch-crf tqdm scikit-learn pandas numpy pythainlp -q

In [2]:
import os, glob, heapq
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
from pythainlp.tokenize import word_tokenize

if hasattr(torch.backends,'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

LABEL_MAP = {0:'B_WORD', 1:'I_WORD', 2:'E_WORD'}
SEQ_LEN = 256
BATCH_SIZE = 128

Device: mps


In [3]:
def get_char_type(c):
    cp = ord(c)
    if 0x0E01<=cp<=0x0E2E: return 1
    if cp in (0x0E30,0x0E32,0x0E33,0x0E40,0x0E41,0x0E42,0x0E43,0x0E44): return 2
    if cp in (0x0E31,0x0E34,0x0E35,0x0E36,0x0E37,0x0E38,0x0E39,0x0E47): return 3
    if 0x0E48<=cp<=0x0E4B: return 4
    if cp in (0x0E4C,0x0E4D,0x0E4E,0x0E2F): return 5
    if 0x0E50<=cp<=0x0E59: return 6
    if c.isascii() and c.isalpha(): return 7
    if c.isdigit(): return 8
    return 9

def extract_dict_tags(text):
    words = word_tokenize(text, engine='newmm', keep_whitespace=True)
    dct_tags = []
    for w in words:
        n = len(w)
        if n == 0: continue
        if n == 1: dct_tags.append(0)
        elif n == 2: dct_tags.extend([0, 2])
        else: dct_tags.extend([0] + [1]*(n-2) + [2])
    if len(dct_tags) > len(text): dct_tags = dct_tags[:len(text)]
    while len(dct_tags) < len(text): dct_tags.append(0)
    return dct_tags

def parse_training_file(filepath):
    raw_chars, labels = [], []
    with open(filepath,'r',encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n\r')
            if not line.strip(): continue
            parts = line.split('\t')
            if len(parts)<2: continue
            word = parts[0]
            if word=='_': continue
            wc = list(word); n = len(wc)
            if n==0: continue
            if n==1: raw_chars.append(wc[0]); labels.append(0)
            elif n==2: raw_chars.extend(wc); labels.extend([0,2])
            else: raw_chars.extend(wc); labels.extend([0]+[1]*(n-2)+[2])
    text = ''.join(raw_chars)
    dct_tags = extract_dict_tags(text)
    return raw_chars, labels, dct_tags

all_chars, all_labels, all_dct = [], [], []
for d in ['LST20_Corpus/train','LST20_Corpus/eval','LST20_Corpus/test']:
    if not os.path.exists(d): continue
    files = sorted(glob.glob(os.path.join(d,'*.txt')))
    for f in tqdm(files, desc=os.path.basename(d)):
        c, l, d_tag = parse_training_file(f)
        all_chars.extend(c); all_labels.extend(l); all_dct.extend(d_tag)
print(f'Total: {len(all_chars):,} chars')

train:   0%|          | 0/3794 [00:00<?, ?it/s]

eval:   0%|          | 0/474 [00:00<?, ?it/s]

test:   0%|          | 0/483 [00:00<?, ?it/s]

Total: 11,523,899 chars


In [4]:
cc = {}; [cc.update({c:cc.get(c,0)+1}) for c in all_chars]
vocab = {'<PAD>':0,'<UNK>':1}; [vocab.update({c:len(vocab)}) for c,n in sorted(cc.items(),key=lambda x:-x[1]) if n>=1]

class DS(Dataset):
    def __init__(s,chars,labels,dct_tags,vocab,sl=256,step=None):
        step=step or sl; s.vocab=vocab; s.sl=sl
        s.S=[]; s.L=[]; s.D=[]
        for i in range(0,len(chars),step):
            e=min(i+sl,len(chars))
            if e-i<10: continue
            s.S.append(chars[i:e]); s.L.append(labels[i:e]); s.D.append(dct_tags[i:e])
    def __len__(s): return len(s.S)
    def __getitem__(s,i):
        c=s.S[i]; l=s.L[i]; d=s.D[i]; n=len(c); p=s.sl-n
        return (torch.tensor([s.vocab.get(x,1) for x in c]+[0]*p),
                torch.tensor([get_char_type(x) for x in c]+[0]*p),
                torch.tensor(list(d)+[0]*p),
                torch.tensor(list(l)+[0]*p),
                torch.tensor([True]*n+[False]*p))

sp=int(len(all_chars)*0.95)
tl=DataLoader(DS(all_chars[:sp], all_labels[:sp], all_dct[:sp], vocab, SEQ_LEN, step=128), batch_size=BATCH_SIZE, shuffle=True)
vl=DataLoader(DS(all_chars[sp:], all_labels[sp:], all_dct[sp:], vocab, SEQ_LEN, step=SEQ_LEN), batch_size=BATCH_SIZE)


In [5]:
class M(nn.Module):
    def __init__(s, vs, nt=10, nd=3, ed=128, td=16, dd=16, h=256, nl=3, ly=3, dr=0.3):
        super().__init__()
        s.ce=nn.Embedding(vs,ed,padding_idx=0)
        s.te=nn.Embedding(nt,td)
        s.de=nn.Embedding(nd+1,dd)
        s.lstm=nn.LSTM(ed+td+dd, h, ly, bidirectional=True, batch_first=True, dropout=dr)
        s.fc=nn.Linear(h*2,nl)
        s.crf=CRF(nl,batch_first=True)
        s.drop=nn.Dropout(dr)
    def _e(s,ci,ct,cd):
        x=torch.cat([s.ce(ci), s.te(ct), s.de(cd)], -1)
        x=s.drop(x); x,_=s.lstm(x); return s.fc(s.drop(x))
    def forward(s,ci,ct,cd,lb,mk): return -s.crf(s._e(ci,ct,cd),lb,mask=mk,reduction='mean')
    def predict(s,ci,ct,cd,mk): return s.crf.decode(s._e(ci,ct,cd),mask=mk)

model=M(len(vocab)).to(device)

In [ ]:
opt=torch.optim.Adam(model.parameters(), lr=0.001)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=2, factor=0.5)

# ★ ระบบเก็บสมองที่เก่งที่สุด 3 อันดับแรก (Top-3 Checkpoints)
top_k = 3
top_checkpoints = [] # List of (f1, weights)

for ep in range(15):
    model.train(); tl_sum=0
    for b in tqdm(tl, desc=f'Epoch {ep+1}/15', leave=False):
        ci,ct,cd,lb,mk=[x.to(device) for x in b]
        loss=model(ci,ct,cd,lb,mk); opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step(); tl_sum+=loss.item()
    
    model.eval(); ap,at=[],[]
    with torch.no_grad():
        for b in vl:
            ci,ct,cd,lb,mk=[x.to(device) for x in b]
            ps=model.predict(ci,ct,cd,mk)
            for p,t,m in zip(ps,lb.cpu().numpy(),mk.cpu().numpy()):
                n=int(m.sum()); ap.extend(p[:n]); at.extend(t[:n].tolist())
    
    f1=f1_score(at,ap,average='macro'); sch.step(tl_sum/len(tl))
    print(f'Epoch {ep+1}: F1={f1:.4f}')
    
    # เก็บลง Heap (เก็บ Top-3)
    st = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    if len(top_checkpoints) < top_k:
        heapq.heappush(top_checkpoints, (f1, st))
    else:
        if f1 > top_checkpoints[0][0]:
            heapq.heapreplace(top_checkpoints, (f1, st))

print('\n✅ Training finished. Top F1 scores:', [f[0] for f in top_checkpoints])

Epoch 1/15:   0%|          | 0/669 [00:00<?, ?it/s]

Epoch 1: F1=0.9843


Epoch 2/15:   0%|          | 0/669 [00:00<?, ?it/s]

Epoch 2: F1=0.9863


Epoch 3/15:   0%|          | 0/669 [00:00<?, ?it/s]

...

Epoch 8/15:   0%|          | 0/669 [00:00<?, ?it/s]

Epoch 8: F1=0.9866


Epoch 9/15:   0%|          | 0/669 [00:00<?, ?it/s]

Epoch 9: F1=0.9873


Epoch 10/15:   0%|          | 0/669 [00:00<?, ?it/s]

In [ ]:
with open('LST20_Corpus/ws_test.txt','r',encoding='utf-8') as f: 
    test_text=f.read()
test_dict_tags = extract_dict_tags(test_text)

test_chars, test_ids, test_dct = [], [], []
for i,c in enumerate(test_text):
    if c in ' \t\n\r': continue
    test_chars.append(c); test_ids.append(i+1); test_dct.append(test_dict_tags[i])

STEP, nc = 32, len(test_chars)
MARGIN = (SEQ_LEN - STEP) // 2

# ★ Ensemble Inference: ทำนายด้วย 3 โมเดลแล้วโหวต
all_predictions = []
for f1, weights in top_checkpoints:
    print(f'Predicting with model (F1={f1:.4f})...')
    model.load_state_dict(weights); model.to(device); model.eval()
    fp = [None]*nc
    for st in tqdm(range(0,nc,STEP), leave=False):
        end = min(st+SEQ_LEN, nc)
        seg=test_chars[st:end]; sdct=test_dct[st:end]
        n=len(seg); pad=SEQ_LEN-n
        ci=[vocab.get(ch,1) for ch in seg]+[0]*pad
        ct=[get_char_type(ch) for ch in seg]+[0]*pad
        cd=sdct+[0]*pad; mk=[True]*n+[False]*pad
        with torch.no_grad():
            ps=model.predict(torch.tensor([ci]).to(device), torch.tensor([ct]).to(device), 
                             torch.tensor([cd]).to(device), torch.tensor([mk]).to(device))[0][:n]
        ks,ke = (0,min(STEP+MARGIN,n)) if st==0 else ((min(MARGIN,n),n) if end>=nc else (MARGIN,min(MARGIN+STEP,n)))
        for j in range(ks,ke): 
            if st+j<nc: fp[st+j]=ps[j]
    all_predictions.append(fp)

# --- 🗳️ Voting & Rule Engine ---
final_fp = []
for i in range(nc):
    # Majority Vote
    votes = [p[i] for p in all_predictions]
    winner = max(set(votes), key=votes.count)
    
    # กฎภาษาไทย
    cp = ord(test_chars[i])
    if ((cp==0x0E31) or (0x0E34<=cp<=0x0E3A) or (0x0E47<=cp<=0x0E4E)) and winner==0: winner=1
    
    # กฎเลข/อังกฤษ: ห้ามตัดขึ้นต้นคำถ้าตัวก่อนหน้าเป็นเลข/อังกฤษเหมือนกัน (รักษาความต่อเนื่อง)
    if i > 0 and winner == 0:
        prev_type = get_char_type(test_chars[i-1])
        curr_type = get_char_type(test_chars[i])
        if prev_type in (7,8) and curr_type == prev_type:
            winner = 1 # เปลี่ยน B เป็น I
            
    final_fp.append(winner)

sub=pd.DataFrame({'Id':test_ids,'Predicted':[LABEL_MAP[p] for p in final_fp]})
sub.to_csv('submission.csv',index=False)
print('✅ submission.csv saved using TOP-3 ENSEMBLE!')